# Giai đoạn 2: Khám phá quy trình hướng đối tượng (OCPM Discovery)

Sau khi đã có OCEL, chúng ta sẽ:
1. **Discover Object-Centric Petri Net (OCPN)** — Mô hình quy trình tổng thể
2. **Vẽ Object Interaction Graph** — Đối tượng nào tương tác với nhau ở đâu?
3. **Phân tích per-object trace** — Trace riêng của Application vs Offer

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pm4py
import warnings
warnings.filterwarnings('ignore')

print(f"pm4py version: {pm4py.__version__}")

## 1. Tải OCEL đã chuyển đổi

In [ ]:
# Tải OCEL từ file
print("⏳ Đang tải OCEL...")
try:
    ocel = pm4py.read.read_ocel_json('output/bpi2017_ocel.jsonocel')
    print("✅ Tải từ OCEL JSON thành công!")
except Exception as e:
    print(f"⚠️  Không tải được JSON ({e}), thử từ CSV...")
    # Fallback: tải từ CSV
    events_df   = pd.read_csv('output/bpi2017_events.csv')
    objects_df  = pd.read_csv('output/bpi2017_objects.csv')
    relations_df = pd.read_csv('output/bpi2017_relations.csv')
    print("✅ Tải từ CSV thành công!")
    events_df['ocel:timestamp'] = pd.to_datetime(events_df['ocel:timestamp'], utc=True, errors='coerce')
    relations_df['ocel:timestamp'] = pd.to_datetime(relations_df['ocel:timestamp'], utc=True, errors='coerce')

    from pm4py.objects.ocel.obj import OCEL
    # Tái tạo omap
    from ast import literal_eval
    omap = relations_df.groupby('ocel:eid').apply(
        lambda g: [{'objectId': r['ocel:oid'], 'objectType': r['ocel:type']}
                   for _, r in g.iterrows()]
    ).reset_index(name='ocel:omap')
    events_merged = events_df.merge(omap, on='ocel:eid', how='left')
    ocel = OCEL(events=events_merged, objects=objects_df, relations=relations_df)

print(f"\n📊 OCEL Stats:")
print(f"  • Sự kiện: {len(ocel.events):,}")
print(f"  • Đối tượng: {len(ocel.objects):,}")
print(f"  • Quan hệ E-O: {len(ocel.relations):,}")

## 2. Thống kê hoạt động theo loại đối tượng

In [ ]:
# Phân tích hoạt động theo object type
relations_df = pd.read_csv('output/bpi2017_relations.csv')
relations_df['ocel:timestamp'] = pd.to_datetime(relations_df['ocel:timestamp'], utc=True, errors='coerce')

activity_by_type = relations_df.groupby(['ocel:activity', 'ocel:type']).size().unstack(fill_value=0)

print("📋 Số sự kiện theo Activity × Object Type:")
display(activity_by_type)

In [ ]:
# Visualize: Heatmap Activity × Object Type
fig, ax = plt.subplots(figsize=(10, 8))

# Chuẩn hóa log scale để thấy rõ
data_log = np.log1p(activity_by_type.values)

im = ax.imshow(data_log, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(activity_by_type.columns)))
ax.set_xticklabels(activity_by_type.columns, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(len(activity_by_type.index)))
ax.set_yticklabels(activity_by_type.index, fontsize=10)

# Thêm số vào ô
for i in range(len(activity_by_type.index)):
    for j in range(len(activity_by_type.columns)):
        val = activity_by_type.values[i, j]
        if val > 0:
            text_color = 'white' if data_log[i, j] > data_log.max() * 0.6 else 'black'
            ax.text(j, i, f'{val:,}', ha='center', va='center',
                    fontsize=9, color=text_color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Log(count+1)')
ax.set_title('Mô hình tương tác: Hoạt động × Loại Đối tượng (OCPM View)', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('output/activity_object_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: output/activity_object_heatmap.png")

## 3. Directly-Follows Graph (DFG) theo từng Object Type

Thay vì 1 DFG "bùi nhùi" cho toàn bộ log, OCPM cho phép tách riêng DFG theo từng object type.

In [ ]:
# DFG cho Application objects
app_events = relations_df[relations_df['ocel:type'] == 'application'].copy()
app_events_sorted = app_events.sort_values(['ocel:oid', 'ocel:timestamp'])

# Tính Directly-Follows
app_events_sorted['next_activity'] = app_events_sorted.groupby('ocel:oid')['ocel:activity'].shift(-1)
dfg_app = app_events_sorted.dropna(subset=['next_activity']).groupby(
    ['ocel:activity', 'next_activity']).size().reset_index(name='count')
dfg_app = dfg_app.sort_values('count', ascending=False).head(15)

print("📊 Top 15 Directly-Follows (Application perspective):")
display(dfg_app)

In [ ]:
# DFG cho Offer objects
offer_events = relations_df[relations_df['ocel:type'] == 'offer'].copy()
offer_events_sorted = offer_events.sort_values(['ocel:oid', 'ocel:timestamp'])

offer_events_sorted['next_activity'] = offer_events_sorted.groupby('ocel:oid')['ocel:activity'].shift(-1)
dfg_offer = offer_events_sorted.dropna(subset=['next_activity']).groupby(
    ['ocel:activity', 'next_activity']).size().reset_index(name='count')
dfg_offer = dfg_offer.sort_values('count', ascending=False).head(15)

print("📊 Top 15 Directly-Follows (Offer perspective):")
display(dfg_offer)

In [ ]:
# Visualize DFG cả 2 perspectives
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

def plot_dfg_bar(ax, dfg, title, color):
    dfg_top = dfg.head(10).copy()
    labels = [f"{r['ocel:activity'].split()[-1]}\n→ {r['next_activity'].split()[-1]}"
              for _, r in dfg_top.iterrows()]
    bars = ax.barh(range(len(labels)), dfg_top['count'].values,
                   color=color, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Số lần xuất hiện', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    for bar in bars:
        width = bar.get_width()
        ax.text(width * 1.01, bar.get_y() + bar.get_height()/2,
                f'{int(width):,}', va='center', fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plot_dfg_bar(axes[0], dfg_app,   'DFG — Góc nhìn Application', 'steelblue')
plot_dfg_bar(axes[1], dfg_offer, 'DFG — Góc nhìn Offer',       'coral')

plt.tight_layout()
plt.savefig('output/dfg_by_object_type.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: output/dfg_by_object_type.png")

## 4. Object Lifecycle Analysis — Vòng đời của Offer

In [ ]:
# Phân tích kết cục của Offer (Lifecycle endpoint)
offer_last_event = offer_events_sorted.groupby('ocel:oid')['ocel:activity'].last().reset_index()
offer_last_event.columns = ['OfferID', 'final_activity']

outcome_counts = offer_last_event['final_activity'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if 'Accept' in x else '#e74c3c' if 'Refus' in x
          else '#f39c12' if 'Cancel' in x else '#3498db' for x in outcome_counts.index]
bars = ax.bar(outcome_counts.index, outcome_counts.values, color=colors, edgecolor='white')
ax.set_title('Kết cục cuối cùng của các Offer (Object Lifecycle)', fontsize=13)
ax.set_xlabel('Hoạt động cuối cùng')
ax.set_ylabel('Số lượng Offer')
ax.tick_params(axis='x', rotation=30)

for bar in bars:
    height = bar.get_height()
    pct = height / len(offer_last_event) * 100
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/offer_lifecycle_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("🏁 OCPM Discovery hoàn tất!")
print("=" * 60)
print("\nFile output:")
print("  📊 output/activity_object_heatmap.png")
print("  📊 output/dfg_by_object_type.png")
print("  📊 output/offer_lifecycle_outcomes.png")
print("\n▶️  Chạy tiếp: ocpm_business_insights.ipynb")